In [1]:
import pandas as pd
from h3 import h3
import glob
import numpy as np
from pandas import merge
import json
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import shape
from datetime import datetime, timedelta
import seaborn as sns

# Part 1: Derive Relocation Rules from Historical Data

In [11]:
# Pandas Config
pd.set_option('display.max_columns', None)
pd.options.display.max_rows = 200

# ShareNow Data (results from Prepare_Trip_Data)
VEHICLE_TRIP = '/Users/demircanm/sciebo/IS3/Research_Projects/HeterogeneousFleetManagement/01_Fleet_Sizing/0_Data/Berlin/shared_vehicle_trips_berlin.csv'

# TIMEZONE
TIMEZONE = 'Europe/Berlin'


In [12]:
# Read Trips
trips = pd.read_csv(VEHICLE_TRIP)

/Users/demircanm/opt/anaconda3/envs/SharedMobility/lib/python3.7/site-packages/IPython/core/interactiveshell.py:3146: DtypeWarning: Columns (4,21,22) have mixed types.Specify dtype option on import or set low_memory=False.
  interactivity=interactivity, compiler=compiler, result=result)


In [66]:
# Clean Vehicle Type
trips["vehicleType"] = trips["vehicleType"].replace('bycicle', 'bicycle')

In [68]:
hex_start_column, hex_end_column = "h3_start_7", "h3_end_7"
day_of_week_column = "is_holiday"
date_column = "date_start"
vehicle_type = "vehicleType"



In [69]:
# Count outflow trips in spatio temporal resolution
outflow = trips.groupby([date_column, vehicle_type, day_of_week_column, hex_start_column]).agg({"id": "count"}).reset_index()

outflow.rename(columns={"id":"count"}, inplace=True)

# Reduce to WorkingDay/Non-Working-Day
outflow = outflow.groupby([day_of_week_column, vehicle_type, hex_start_column]).agg({"count":"median"})

In [70]:
# Count outflow trips in spatio temporal resolution
inflow = trips.groupby([date_column, vehicle_type, day_of_week_column, hex_end_column]).agg({"id": "count"}).reset_index()

inflow.rename(columns={"id":"count"}, inplace=True)

# Reduce to WorkingDay/Non-Working-Day
inflow = inflow.groupby([day_of_week_column, vehicle_type, hex_end_column]).agg({"count":"median"})

In [73]:
# Consider only hexagons that have data for each temporal data bucket
hexagon_to_consider = trips.groupby([hex_start_column]).agg({"id": "count"}).reset_index()
hexagon_to_consider["key"] = 1 # For Cross Join Later

temporal_dimension_to_consider = trips.groupby([day_of_week_column]).agg({"id": "count"}).reset_index()
temporal_dimension_to_consider["key"] = 1 # For Cross Join Later

vehicle_type_to_consider = trips.groupby([vehicle_type]).agg({"id": "count"}).reset_index()
vehicle_type_to_consider["key"] = 1 # For Cross Join Later

result = hexagon_to_consider.merge(temporal_dimension_to_consider, on="key", how="inner")
result = result.merge(vehicle_type_to_consider, on="key", how="inner")

result = result.drop(columns={"id_x", "key", "id_y", "id"})

In [74]:
result

,h3_start_7,is_holiday,vehicleType
0,871f18840ffffff,False,bicycle
1,871f18840ffffff,False,car
2,871f18840ffffff,False,kick scooter
3,871f18840ffffff,False,scooter
4,871f18840ffffff,True,bicycle
...,...,...,...
1059,871f1d591ffffff,False,scooter
1060,871f1d591ffffff,True,bicycle
1061,871f1d591ffffff,True,car
1062,871f1d591ffffff,True,kick scooter


In [75]:
# Merge Inflow
result = result.merge(inflow, left_on=[hex_start_column, vehicle_type, day_of_week_column], right_on=[hex_end_column, vehicle_type, day_of_week_column], how = "left")

result = result.rename(columns={"count":"inflow"})

# Merge Outflow
result = result.merge(outflow, left_on=[hex_start_column, vehicle_type, day_of_week_column], right_on=[hex_start_column, vehicle_type, day_of_week_column], how = "left")

result = result.rename(columns={"count":"outflow"})

# Fill NA
result = result.fillna(0)

In [76]:
result["diff"] = result["inflow"] - result["outflow"]

In [81]:
# Not used in Simulation so far...
result.to_csv('inflow_outflow_hex_7_daily.csv', index = False)

# Part 2: Derive Relocation Rules from Simulation Data

In [11]:
# Pandas Config
pd.set_option('display.max_columns', None)
pd.options.display.max_rows = 200

# ShareNow Data (results from Prepare_Trip_Data)
TRANSACTIONS = '../../data/output/transactions.csv'

# TIMEZONE
TIMEZONE = 'Europe/Berlin'

In [12]:
# Read Trips
trips = pd.read_csv(TRANSACTIONS)

In [13]:
trips

,origin,destination,time_start,time_end,price,distance,duration,selected_vehicle_mode,state,SharedCar,SharedKickScooter,SharedScooter
0,871f1d4d2ffffff,871f1d4d0ffffff,1.594855,1.594855,0.000000,1173.466260,0.000000,None,Missed,0,1,0
1,871f18b24ffffff,871f18b24ffffff,0.151785,3.987756,0.127048,222.888665,0.668673,SharedKickScooter,Fulfilled,0,1,0
2,871f18b20ffffff,871f18b24ffffff,4.768195,4.768195,0.000000,2507.841293,0.000000,None,Missed,1,0,1
3,871f1d4d4ffffff,871f1d4d4ffffff,0.536715,5.867576,0.625913,1098.081536,3.294278,SharedKickScooter,Fulfilled,0,1,0
4,871f1d48bffffff,871f1d48bffffff,0.337884,5.936408,0.840016,1826.122138,3.652244,SharedScooter,Fulfilled,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...
15461,871f1d4d3ffffff,871f18b21ffffff,1431.970644,1439.641240,1.353442,4337.936335,5.205544,SharedCar,Fulfilled,1,0,1
15462,871f1d4d2ffffff,871f18b21ffffff,1439.687134,1439.687134,0.000000,1110.113453,0.000000,None,Missed,0,1,0
15463,871f1d483ffffff,871f1d48bffffff,1426.907392,1439.829643,1.682977,5394.136130,6.472989,SharedCar,Fulfilled,1,0,0
15464,871f18b25ffffff,871f18b24ffffff,1429.904112,1439.908304,1.311197,4202.536751,5.043064,SharedCar,Fulfilled,1,0,1


In [221]:
ordered_available_vehicle = ["SharedKickScooter", "SharedScooter", "SharedCar"]
# Rearrange Columns according to Vehicle Preferences
cols = trips.columns.tolist()
cols = cols[:-(len(ordered_available_vehicle))] + ordered_available_vehicle
cols

# Rearrange Columns
trips = trips[cols]

trips["cheapest_vehicle_mode"] = ''

for i in range(len(ordered_available_vehicle)):
    query = '{} == {}'.format(ordered_available_vehicle[i], 1)
    for k in range(i):
        query += ' & {} == {}'.format(ordered_available_vehicle[k], 0)
    
    trips.loc[trips.query(query).index, "cheapest_vehicle_mode"] = ordered_available_vehicle[i]

In [134]:
# (All) Hexagons
hexagons = pd.unique(trips[["origin", "destination"]].values.ravel('K'))
hexagons = pd.DataFrame(hexagons, columns=["hex_id"])

In [272]:
# Calculate Expected Outflow 
outflow = trips.groupby(["origin", "state", "cheapest_vehicle_mode"]).agg({"destination":"count"})

# Unstack Vehicle Type Level 
outflow = outflow.unstack()
vehicle_columns = list(outflow["destination"].columns.values)
# Reset Multi-Index to Flat Index
outflow = outflow.reset_index()
outflow.columns = ["origin", "state"] + vehicle_columns

# to ensure that "fulfilled" comes before "missed" during unstacking
outflow = outflow.sort_values(["origin", "state"])

# Unstack State Column
outflow = outflow.set_index(["origin", "state"])
outflow = outflow.unstack().reset_index()

# Arrange Column Names
unfolded_cols = ["hex_id"] 
for vehicle in vehicle_columns:
    unfolded_cols = unfolded_cols + ["OutFlow_Fulfilled_{}".format(vehicle), "OutFlow_Missed_{}".format(vehicle)]

# Overwrite Column Names
outflow.columns = unfolded_cols

In [282]:
# Calculate Expected Inflow
inflow = trips.groupby(["destination", "state", "cheapest_vehicle_mode"]).agg({"origin":"count"})

# Unstack Vehicle Type Level
inflow = inflow.unstack()
base_column = ["destination", "state"]
vehicle_columns = list(inflow["origin"].columns.values)

# Reset Multi-Index to Flat Index (keep vehicle column order)
inflow = inflow.reset_index()
inflow.columns = ["hex_id", "state"] + ["Inflow_{}".format(vehicle) for vehicle in vehicle_columns]

# to ensure that "fulfilled" comes before "missed" during unstacking
inflow = inflow.sort_values(["hex_id", "state"])

# Inflow, we just need fulfilled trips to estimate expected inflow
inflow = inflow[inflow["state"]=="Fulfilled"]
inflow = inflow.drop(columns={"state"})
inflow = inflow.reset_index()

In [284]:
# Put together expected outflow and inflow
result = hexagons.merge(inflow, on = ["hex_id"], how = "left", suffixes = ("",""))
result = result.merge(outflow, on = ["hex_id"], how = "left", suffixes = ("_inflow", "_outflow"))
result.fillna(0, inplace=True)

In [289]:
# Calculate Delta (Expected Inflow - Expected Outflow - Missed Demand)
for vehicle in ordered_available_vehicle:
    result["Delta_{}".format(vehicle)] = result["Inflow_{}".format(vehicle)] - result["OutFlow_Fulfilled_{}".format(vehicle)] - result["OutFlow_Missed_{}".format(vehicle)]

In [298]:
estimate_delta_demand(trips)

/Users/demircanm/opt/anaconda3/envs/SharedMobility/lib/python3.7/site-packages/ipykernel_launcher.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  # This is added back by InteractiveShellApp.init_path()
/Users/demircanm/opt/anaconda3/envs/SharedMobility/lib/python3.7/site-packages/pandas/core/indexing.py:1765: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  isetter(loc, value)
/Users/demircanm/opt/anaconda3/envs/SharedMobility/lib/python3.7/site-packages/pandas/core/indexing.py:1765: SettingWithCopyWarnin

,hex_id,index,Inflow_SharedCar,Inflow_SharedKickScooter,Inflow_SharedScooter,OutFlow_Fulfilled_SharedCar,OutFlow_Missed_SharedCar,OutFlow_Fulfilled_SharedKickScooter,OutFlow_Missed_SharedKickScooter,OutFlow_Fulfilled_SharedScooter,OutFlow_Missed_SharedScooter,Delta_SharedKickScooter,Delta_SharedScooter,Delta_SharedCar
0,871f1d4d2ffffff,112,148.0,72.0,229.0,130.0,69.0,75.0,57.0,240.0,2.0,-60.0,-13.0,-51.0
1,871f18b24ffffff,20,111.0,71.0,180.0,135.0,51.0,65.0,37.0,182.0,19.0,-31.0,-21.0,-75.0
2,871f18b20ffffff,12,24.0,1.0,21.0,12.0,7.0,1.0,5.0,17.0,12.0,-5.0,-8.0,5.0
3,871f1d4d4ffffff,116,89.0,64.0,192.0,98.0,18.0,61.0,33.0,157.0,12.0,-30.0,23.0,-27.0
4,871f1d48bffffff,64,201.0,298.0,381.0,216.0,29.0,313.0,19.0,384.0,19.0,-34.0,-22.0,-44.0
5,871f1d488ffffff,58,158.0,256.0,303.0,183.0,18.0,270.0,37.0,291.0,17.0,-51.0,-5.0,-43.0
6,871f1d489ffffff,60,200.0,441.0,476.0,218.0,39.0,444.0,11.0,505.0,3.0,-14.0,-32.0,-57.0
7,871f1d48affffff,62,135.0,173.0,281.0,174.0,22.0,188.0,23.0,267.0,17.0,-38.0,-3.0,-61.0
8,871f1d4f3ffffff,136,32.0,4.0,31.0,28.0,13.0,1.0,12.0,16.0,9.0,-9.0,6.0,-9.0
9,871f1d498ffffff,74,71.0,18.0,80.0,60.0,7.0,22.0,29.0,67.0,26.0,-33.0,-13.0,4.0


In [297]:
def estimate_delta_demand(transactions, ordered_available_vehicle = ["SharedKickScooter", "SharedScooter", "SharedCar"]):
    """
        Compute the expected demand in a hexagon based on collected rentral transactions from the simulation. 
        
        Transactions: Transaction Records from the Payment Interface
        Ordered_Available_Vehicle: List with Available Vehicles and Natural Ordering of Preference in case of neutrality
            i.e., the following hodls 
                ordered_available_vehicle[i] = 1 and ordered_available_vehicle[k] = 1
                            
                                        k > i 
                                        
                => 
                
                Demand is counted for: ordered_available_vehicle[i] as we assume preference order i << k 
        
        We calculate for each vehicle type -- based on the previous assumtion -- the delta demand:
        
                E[Delta Demand| Hexagon ID] = E[Inflow | Hexagon ID] - E[Outflow s| Hexagon ID] - E[Missed Trips | Hexagon ID]
    """
    ordered_available_vehicle = ["SharedKickScooter", "SharedScooter", "SharedCar"]
    # Rearrange Columns according to Vehicle Preferences
    cols = transactions.columns.tolist()
    cols = cols[:-(len(ordered_available_vehicle))] + ordered_available_vehicle
    cols

    # Rearrange Columns
    transactions = transactions[cols]

    transactions["cheapest_vehicle_mode"] = ''

    for i in range(len(ordered_available_vehicle)):
        query = '{} == {}'.format(ordered_available_vehicle[i], 1)
        for k in range(i):
            query += ' & {} == {}'.format(ordered_available_vehicle[k], 0)

        transactions.loc[transactions.query(query).index, "cheapest_vehicle_mode"] = ordered_available_vehicle[i]    
        

    # (All) Hexagons
    hexagons = pd.unique(transactions[["origin", "destination"]].values.ravel('K'))
    hexagons = pd.DataFrame(hexagons, columns=["hex_id"])
    
    # Calculate Expected Outflow 
    outflow = transactions.groupby(["origin", "state", "cheapest_vehicle_mode"]).agg({"destination":"count"})

    # Unstack Vehicle Type Level 
    outflow = outflow.unstack()
    vehicle_columns = list(outflow["destination"].columns.values)
    # Reset Multi-Index to Flat Index
    outflow = outflow.reset_index()
    outflow.columns = ["origin", "state"] + vehicle_columns

    # to ensure that "fulfilled" comes before "missed" during unstacking
    outflow = outflow.sort_values(["origin", "state"])

    # Unstack State Column
    outflow = outflow.set_index(["origin", "state"])
    outflow = outflow.unstack().reset_index()

    # Arrange Column Names
    unfolded_cols = ["hex_id"] 
    for vehicle in vehicle_columns:
        unfolded_cols = unfolded_cols + ["OutFlow_Fulfilled_{}".format(vehicle), "OutFlow_Missed_{}".format(vehicle)]

    # Overwrite Column Names
    outflow.columns = unfolded_cols
    
    # Calculate Expected Inflow
    inflow = transactions.groupby(["destination", "state", "cheapest_vehicle_mode"]).agg({"origin":"count"})

    # Unstack Vehicle Type Level
    inflow = inflow.unstack()
    base_column = ["destination", "state"]
    vehicle_columns = list(inflow["origin"].columns.values)

    # Reset Multi-Index to Flat Index (keep vehicle column order)
    inflow = inflow.reset_index()
    inflow.columns = ["hex_id", "state"] + ["Inflow_{}".format(vehicle) for vehicle in vehicle_columns]

    # to ensure that "fulfilled" comes before "missed" during unstacking
    inflow = inflow.sort_values(["hex_id", "state"])

    # Inflow, we just need fulfilled trips to estimate expected inflow
    inflow = inflow[inflow["state"]=="Fulfilled"]
    inflow = inflow.drop(columns={"state"})
    inflow = inflow.reset_index()
    
    # Put together expected outflow and inflow
    result = hexagons.merge(inflow, on = ["hex_id"], how = "left", suffixes = ("",""))
    result = result.merge(outflow, on = ["hex_id"], how = "left", suffixes = ("_inflow", "_outflow"))
    result.fillna(0, inplace=True)
    
    # Calculate Delta (Expected Inflow - Expected Outflow - Missed Demand)
    for vehicle in ordered_available_vehicle:
        result["Delta_{}".format(vehicle)] = result["Inflow_{}".format(vehicle)] - result["OutFlow_Fulfilled_{}".format(vehicle)] - result["OutFlow_Missed_{}".format(vehicle)]
        
    return result